# LFC WAVE velocity residuals (`get_wavelength_solution`)

Inject known LFC wavelengths on BSPEC fiber 2, evaluate
`SimulateRateMap.get_wavelength_solution` at each deposit row, and report
Δv = c · (λ_WAVE − λ_inj) / ⟨λ⟩ in cm/s.

Writes `lfc_velocity_residuals.pdf` under `OUT`.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

from hispecdrp.primitives.utils.spectral_utils import delta_wave_to_delta_vel
from hispecdrp.simulations.e2e_pipeline import (
    expected_feature_xy_oriented,
    get_sim_hdf_trace_grids,
)
from hispecdrp.simulations.simulate_rate_map import SimulateRateMap

OUT = Path("/Users/ckilpatrick/hispec_lfc_wavesol_refit")
OUT.mkdir(parents=True, exist_ok=True)

SPECTROGRAPH, FIBER = "BSPEC", 2
N_LINES = 50
EDGE = 0.05
PHI = (np.sqrt(5.0) - 1.0) / 2.0  # golden-ratio phase stagger across orders

grids = get_sim_hdf_trace_grids(SPECTROGRAPH)
orders = sorted(grids[FIBER].keys())
pix = np.arange(4096, dtype=np.float64)

# Staggered line list in spectral-row space → deposit wavelengths
line_list = {}
for i, od in enumerate(orders):
    w = np.asarray(grids[FIBER][od]["wavelength"], dtype=np.float64)
    ok = np.isfinite(w)
    lo, hi = np.percentile(pix[ok], [100 * EDGE, 100 * (1 - EDGE)])
    rows = lo + (np.arange(N_LINES, dtype=np.float64) + (i * PHI) % 1.0) * (hi - lo) / N_LINES
    line_list[od] = np.interp(rows, pix, w)

wave = SimulateRateMap().get_wavelength_solution(spectrograph=SPECTROGRAPH)[FIBER]

per_order = {}
for od, lams in line_list.items():
    rows, _ = expected_feature_xy_oriented(SPECTROGRAPH, FIBER, od, lams)
    ok = np.isfinite(rows) & np.isfinite(lams)
    if ok.sum() < 5:
        continue
    r, lam_inj = rows[ok], lams[ok]
    lam_rec = np.interp(r, pix, np.asarray(wave[od]["wavelength"], dtype=np.float64))
    lam_mid = float(np.mean(lam_inj))
    dvel = np.asarray(delta_wave_to_delta_vel(lam_rec - lam_inj, lam_mid), float) * 100.0
    med = float(np.median(dvel))
    per_order[od] = dict(
        lam_inj=lam_inj,
        lam_rec=lam_rec,
        dvel=dvel,
        mad=float(np.median(np.abs(dvel - med))),
        bias=med,
        lam_mid=lam_mid,
        n=int(ok.sum()),
    )

ods = sorted(per_order)
mads = np.array([per_order[o]["mad"] for o in ods])
biases = np.array([per_order[o]["bias"] for o in ods])
print(f"orders={len(ods)}  median MAD={np.median(mads):.3f} cm/s  median bias={np.median(biases):.3f} cm/s")


orders=39  median MAD=0.137 cm/s  median bias=-0.462 cm/s


In [2]:
pdf_path = OUT / "lfc_velocity_residuals.pdf"

with PdfPages(pdf_path) as pdf:
    # Summary page
    fig = plt.figure(figsize=(11, 8.5))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1.0], hspace=0.35, wspace=0.28)

    ax0 = fig.add_subplot(gs[0, :])
    ax0.bar(np.arange(len(ods)), mads, color="C0", alpha=0.85)
    ax0.axhline(np.median(mads), color="C3", ls="--", lw=1.5,
                label=f"median MAD = {np.median(mads):.3f} cm/s")
    ax0.axhline(0.14, color="0.35", ls=":", lw=1.0, label="0.14 cm/s")
    ax0.set_xticks(np.arange(len(ods)), [str(o) for o in ods], rotation=90, fontsize=7)
    ax0.set_ylabel("velocity MAD (cm/s)")
    ax0.set_xlabel("echelle order")
    ax0.set_title(
        f"BSPEC fiber {FIBER}: get_wavelength_solution vs LFC injection\n"
        f"N={N_LINES} lines/order  |  Δv = c·(λ_WAVE(row) − λ_inj)/⟨λ⟩  |  ZP-subtracted MAD"
    )
    ax0.legend(loc="upper right")
    ax0.set_ylim(0, max(0.25, 1.3 * float(mads.max())))
    ax0.grid(True, axis="y", alpha=0.3)

    all_dv = np.concatenate([per_order[o]["dvel"] - per_order[o]["bias"] for o in ods])
    ax1 = fig.add_subplot(gs[1, 0])
    ax1.hist(all_dv, bins=60, color="C0", alpha=0.85)
    ax1.axvline(0, color="k", lw=1)
    ax1.set_xlabel("ZP-subtracted residual (cm/s)")
    ax1.set_ylabel("N lines")
    ax1.set_title(f"All lines (N={all_dv.size})")
    ax1.text(
        0.98, 0.95,
        f"median MAD/order = {np.median(mads):.3f} cm/s\n"
        f"global MAD = {np.median(np.abs(all_dv)):.3f} cm/s",
        transform=ax1.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    ax2 = fig.add_subplot(gs[1, 1])
    ax2.plot(ods, biases, "o-", ms=4, color="C1")
    ax2.axhline(0, color="k", lw=0.8)
    ax2.axhline(np.median(biases), color="C3", ls="--",
                label=f"median bias = {np.median(biases):.3f} cm/s")
    ax2.set_xlabel("echelle order")
    ax2.set_ylabel("median Δv (cm/s)")
    ax2.set_title("Per-order zero-point")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    pdf.savefig(fig)
    plt.close(fig)

    # Per-order pages (3 orders each)
    for i0 in range(0, len(ods), 3):
        chunk = ods[i0 : i0 + 3]
        fig, axes = plt.subplots(
            len(chunk), 2, figsize=(11, 2.6 * len(chunk) + 0.5), squeeze=False,
            gridspec_kw={"wspace": 0.28, "hspace": 0.4},
        )
        fig.suptitle(
            "LFC injection: expected vs recovered WAVE (velocity)  |  get_wavelength_solution",
            fontsize=11,
        )
        for row, od in enumerate(chunk):
            d = per_order[od]
            dv, dv0 = d["dvel"], d["dvel"] - d["bias"]
            lam = d["lam_inj"]

            axL = axes[row, 0]
            axL.axhline(0, color="0.5", lw=0.9)
            axL.plot(lam, dv0, ".", ms=5, color="C0", alpha=0.85, label="ZP-subtracted")
            axL.plot(lam, dv, ".", ms=3.5, color="C1", alpha=0.45, label="raw (incl. ZP)")
            axL.set_ylabel("Δv (cm/s)")
            axL.set_xlabel("λ_inj (µm)")
            axL.set_title(
                f"Order {od}  ⟨λ⟩={d['lam_mid']:.4f} µm  "
                f"MAD={d['mad']:.3f} cm/s  bias={d['bias']:.3f} cm/s  n={d['n']}",
                fontsize=9,
            )
            axL.legend(fontsize=7)
            axL.grid(True, alpha=0.25)
            span = max(0.5, 5 * d["mad"], float(np.percentile(np.abs(dv0), 95)))
            axL.set_ylim(-span, span)

            axR = axes[row, 1]
            lam_ref = float(np.median(lam))
            v_inj = np.asarray(delta_wave_to_delta_vel(lam - lam_ref, lam_ref), float) * 100.0
            v_rec = np.asarray(delta_wave_to_delta_vel(d["lam_rec"] - lam_ref, lam_ref), float) * 100.0
            s = np.argsort(v_inj)
            axR.plot(v_inj[s], v_inj[s], "-", color="0.4", lw=1.0, label="1:1")
            axR.plot(v_inj, v_rec, ".", ms=5, color="C0", alpha=0.85, label="recovered")
            axR.set_xlabel("v(λ_inj) − v(⟨λ⟩) (cm/s)")
            axR.set_ylabel("v(λ_WAVE) − v(⟨λ⟩) (cm/s)")
            axR.set_title(f"Order {od}: expected vs recovered velocity", fontsize=9)
            axR.legend(fontsize=7)
            axR.grid(True, alpha=0.25)
            axR.set_aspect("equal", adjustable="datalim")

        fig.tight_layout(rect=[0, 0, 1, 0.97])
        pdf.savefig(fig)
        plt.close(fig)

    pdf.infodict()["Title"] = "LFC WAVE velocity residuals — get_wavelength_solution"

summary = {
    "spectrograph": SPECTROGRAPH,
    "fiber": FIBER,
    "n_lines_per_order": N_LINES,
    "n_orders": len(ods),
    "median_mad_cm_s": float(np.median(mads)),
    "median_bias_cm_s": float(np.median(biases)),
    "global_mad_cm_s": float(np.median(np.abs(all_dv))),
    "pdf": str(pdf_path),
    "per_order_mad_cm_s": {str(o): per_order[o]["mad"] for o in ods},
}
(OUT / "lfc_velocity_residuals.json").write_text(json.dumps(summary, indent=2) + "\n")
print(f"wrote {pdf_path}")
print(json.dumps({k: summary[k] for k in summary if k != "per_order_mad_cm_s"}, indent=2))


/var/folders/_s/tc8h43192s58wb4s6m2lbfcw0000gn/T/ipykernel_84560/2153595020.py:98: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.97])


wrote /Users/ckilpatrick/hispec_lfc_wavesol_refit/lfc_velocity_residuals.pdf
{
  "spectrograph": "BSPEC",
  "fiber": 2,
  "n_lines_per_order": 50,
  "n_orders": 39,
  "median_mad_cm_s": 0.13658757254222098,
  "median_bias_cm_s": -0.4620346773169939,
  "global_mad_cm_s": 0.13716357334594778,
  "pdf": "/Users/ckilpatrick/hispec_lfc_wavesol_refit/lfc_velocity_residuals.pdf"
}
